In [1]:

import pandas as pd
import zipfile
from datetime import datetime, timezone
from pathlib import Path

# 1. Charger les données Bronze
path = Path(
    "/lakehouse/default/Files/bronze/"
    "gtfs_cts/strasbourg/cts_strasbourg_gtfs.zip"
)

with zipfile.ZipFile(path) as z:
    with z.open("stops.txt") as f:
        raw = pd.read_csv(f, dtype=str)

print("Nombre de lignes initial :", len(raw))

# 2. Sélectionner les colonnes
required = ["stop_id", "stop_name", "stop_lat", "stop_lon"]
optional = ["location_type", "parent_station"]

missing = set(required) - set(raw.columns)
if missing:
    raise ValueError(f"Colonnes manquantes : {missing}")

columns = required + [c for c in optional if c in raw.columns]
clean = raw[columns].copy()

# 3. Nettoyage des champs textuels
for col in ["stop_id", "stop_name"]:
    clean[col] = clean[col].fillna("").str.strip()

for col in optional:
    if col in clean.columns:
        clean[col] = clean[col].fillna("").str.strip()

# 4. Conversion des coordonnées
for col in ["stop_lat", "stop_lon"]:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")

# 5. Contrôle de validité
valid = (
    clean["stop_id"].ne("")
    & clean["stop_name"].ne("")
    & clean["stop_lat"].between(-90, 90)
    & clean["stop_lon"].between(-180, 180)
)

rejected = int((~valid).sum())
clean = clean.loc[valid].copy()

# Ne pas fusionner les quais ayant le même nom
duplicates = int(clean["stop_id"].duplicated().sum())
if duplicates:
    raise ValueError(
        f"{duplicates} stop_id dupliqués : vérification nécessaire"
    )

# 6. Ajouter la provenance multi-villes
clean["city"] = "Strasbourg"
clean["territory_id"] = "FR-EMS"
clean["source"] = "CTS_GTFS"
clean["ingested_at_utc"] = datetime.now(
    timezone.utc
).isoformat()

# 7. Transformer en PySpark
spark_stops = spark.createDataFrame(clean)

# 8. Enregistrer dans Silver au format Delta
spark_stops.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.cts_stops")

print("Table Silver créée : silver.cts_stops")
print("Lignes rejetées :", rejected)
print("Arrêts conservés :", spark_stops.count())

display(spark.table("silver.cts_stops").limit(10))


In [2]:

import pandas as pd
import zipfile
from datetime import datetime, timezone

# 1. Chemin du fichier Bronze
path = (
    "/lakehouse/default/Files/bronze/"
    "gtfs_cts/strasbourg/cts_strasbourg_gtfs.zip"
)

# 2. Charger les lignes CTS
with zipfile.ZipFile(path) as z:
    with z.open("routes.txt") as f:
        routes = pd.read_csv(
            f, dtype=str, keep_default_na=False
        )

print("Lignes initiales :", len(routes))

# 3. Colonnes utiles
columns = [
    "route_id",
    "route_short_name",
    "route_long_name",
    "route_type"
]

for col in columns:
    if col not in routes.columns:
        routes[col] = ""

routes = routes[columns].copy()

# 4. Nettoyer les valeurs
for col in columns:
    routes[col] = routes[col].str.strip()

# 5. Vérifier les identifiants
if routes["route_id"].eq("").any():
    raise ValueError("Identifiants de ligne manquants")

if routes["route_id"].duplicated().any():
    raise ValueError("Identifiants de ligne dupliqués")

# 6. Provenance
routes["city"] = "Strasbourg"
routes["territory_id"] = "FR-EMS"
routes["source"] = "CTS_GTFS"
routes["ingested_at_utc"] = datetime.now(
    timezone.utc
).isoformat()

# 7. Convertir en Spark DataFrame
spark_routes = spark.createDataFrame(routes)

# 8. Enregistrer en Delta
spark_routes.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.cts_routes")

# 9. Vérification
print("Table créée : silver.cts_routes")
print("Nombre de lignes :", spark_routes.count())

print("Répartition par type de transport :")
display(
    spark_routes.groupBy("route_type").count()
)

display(
    spark_routes.select(
        "route_id",
        "route_short_name",
        "route_long_name",
        "route_type"
    ).limit(10)
)


In [3]:

import pandas as pd
import zipfile
from datetime import datetime, timezone

path = (
    "/lakehouse/default/Files/bronze/"
    "gtfs_cts/strasbourg/cts_strasbourg_gtfs.zip"
)

# 1. Lire les trajets GTFS
with zipfile.ZipFile(path) as z:
    with z.open("trips.txt") as f:
        trips = pd.read_csv(
            f, dtype=str, keep_default_na=False
        )

print("Nombre de trajets :", len(trips))

# 2. Colonnes obligatoires
required = ["trip_id", "route_id", "service_id"]

missing = set(required) - set(trips.columns)
if missing:
    raise ValueError(f"Colonnes manquantes : {missing}")

# 3. Colonnes facultatives
optional = [
    "trip_headsign",
    "direction_id",
    "shape_id",
    "block_id"
]

for col in optional:
    if col not in trips.columns:
        trips[col] = ""

trips = trips[required + optional].copy()

for col in trips.columns:
    trips[col] = trips[col].str.strip()

# 4. Contrôles de qualité
for col in required:
    if trips[col].eq("").any():
        raise ValueError(f"Valeurs manquantes : {col}")

if trips["trip_id"].duplicated().any():
    raise ValueError("trip_id dupliqués")

# Vérifier les relations avec cts_routes
valid_routes = {
    r["route_id"]
    for r in spark.table("silver.cts_routes")
                  .select("route_id").collect()
}

unknown_routes = set(trips["route_id"]) - valid_routes

if unknown_routes:
    raise ValueError(
        f"Lignes inconnues : {unknown_routes}"
    )

# 5. Ajouter la provenance
trips["city"] = "Strasbourg"
trips["territory_id"] = "FR-EMS"
trips["source"] = "CTS_GTFS"
trips["ingested_at_utc"] = datetime.now(
    timezone.utc
).isoformat()

# 6. Convertir en Spark
spark_trips = spark.createDataFrame(trips)

# 7. Sauvegarder en Delta
spark_trips.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.cts_trips")

print("Table créée : silver.cts_trips")
print("Total trajets :", spark_trips.count())
print(
    "Lignes avec trajets :",
    spark_trips.select("route_id").distinct().count()
)

# 8. Afficher les lignes avec le plus de trajets
display(
    spark_trips.groupBy("route_id")
    .count()
    .orderBy("count", ascending=False)
    .limit(10)
)


In [4]:

from pathlib import Path
from zipfile import ZipFile
import shutil

base = Path(
    "/lakehouse/default/Files/bronze/"
    "gtfs_cts/strasbourg"
)

zip_path = base / "cts_strasbourg_gtfs.zip"
csv_path = base / "stop_times.txt"

with ZipFile(zip_path) as archive:
    matches = [
        n for n in archive.namelist()
        if Path(n).name == "stop_times.txt"
    ]

    if len(matches) != 1:
        raise ValueError("stop_times.txt introuvable ou ambigu")

    with archive.open(matches[0]) as source:
        with open(csv_path, "wb") as destination:
            shutil.copyfileobj(source, destination)

print("Extraction réussie !")
print("Fichier :", csv_path)
print("Taille :", round(csv_path.stat().st_size / 1048576, 2), "Mo")


In [5]:

from pyspark.sql import functions as F

# 1. Charger le CSV avec Spark
df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "FAILFAST")
    .csv(
        "Files/bronze/gtfs_cts/"
        "strasbourg/stop_times.txt"
    )
)

print("Nombre de passages GTFS :", df.count())

# 2. Vérifier les colonnes essentielles
required = [
    "trip_id",
    "stop_id",
    "stop_sequence",
    "arrival_time",
    "departure_time"
]

missing = set(required) - set(df.columns)

if missing:
    raise ValueError(f"Colonnes manquantes : {missing}")

# 3. Nettoyage des chaînes
for col_name in df.columns:
    df = df.withColumn(
        col_name,
        F.trim(F.col(col_name))
    )

# 4. Contrôles des identifiants
for col_name in ["trip_id", "stop_id", "stop_sequence"]:
    invalid = df.filter(
        F.col(col_name).isNull()
        | (F.col(col_name) == "")
    ).count()

    if invalid:
        raise ValueError(
            f"{invalid} valeurs manquantes : {col_name}"
        )

invalid_seq = df.filter(
    ~F.col("stop_sequence").rlike(r"^\d+$")
).count()

if invalid_seq:
    raise ValueError("stop_sequence invalide")

df = df.withColumn(
    "stop_sequence",
    F.col("stop_sequence").cast("int")
)

# 5. Vérifier les références aux trajets
known_trips = spark.table("silver.cts_trips").select("trip_id")

unknown_trips = (
    df.select("trip_id").distinct()
    .join(known_trips, "trip_id", "left_anti")
    .count()
)

# 6. Vérifier les références aux arrêts
known_stops = spark.table("silver.cts_stops").select("stop_id")

unknown_stops = (
    df.select("stop_id").distinct()
    .join(known_stops, "stop_id", "left_anti")
    .count()
)

# 7. Vérifier l'unicité (trip_id, stop_sequence)
duplicates = (
    df.groupBy("trip_id", "stop_sequence")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Trajets inconnus :", unknown_trips)
print("Arrêts inconnus :", unknown_stops)
print("Séquences dupliquées :", duplicates)

if unknown_trips or unknown_stops or duplicates:
    raise ValueError(
        "Anomalies GTFS : corriger avant écriture Silver"
    )

# 8. Ajouter les métadonnées territoriales
df = (
    df.withColumn("city", F.lit("Strasbourg"))
      .withColumn("territory_id", F.lit("FR-EMS"))
      .withColumn("source", F.lit("CTS_GTFS"))
      .withColumn("ingested_at_utc", F.current_timestamp())
)

# 9. Écrire la table Delta
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.cts_stop_times")
)

print("Table créée : silver.cts_stop_times")

# 10. Statistiques
result = spark.table("silver.cts_stop_times")

print("Total des passages :", result.count())
print(
    "Trajets distincts :",
    result.select("trip_id").distinct().count()
)
print(
    "Arrêts distincts desservis :",
    result.select("stop_id").distinct().count()
)

display(
    result.select(
        "trip_id",
        "stop_id",
        "arrival_time",
        "departure_time",
        "stop_sequence"
    ).limit(10)
)
